# Layer 5-7: Backtest Deep Dive Template

This notebook demonstrates an ADR-017 strategy deep dive workflow using the modern DataLoader + VectorizedBacktester stack.
It aligns with the latest reporting standards and includes the rich MFE/MAE analysis.

In [ ]:
import os
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

def find_repo_root(start: Path) -> Path:
    target = Path("scripts/trading_framework/config/sessions.yaml")
    for p in [start, *start.parents]:
        if (p / target).exists():
            return p
    raise FileNotFoundError("Could not locate repo root containing sessions.yaml")

ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(ROOT)
CONFIG_PATH = ROOT / "scripts" / "trading_framework" / "config" / "sessions.yaml"

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from scripts.trading_framework.config.config_loader import load_config
from scripts.libs.data.loader import DataLoader
from scripts.libs.features.feature_registry import FeatureRegistry
from scripts.strategies.reversal.core.box_reversion import BoxReversionStrategy
from scripts.trading_framework.core.backtest_engine import VectorizedBacktester
from scripts.trading_framework.reporting.reporter import QuantReporter
from scripts.trading_framework.core.mfe_mae import compute_mfe_mae_rich, summarize_mfe_mae_rich
from scripts.trading_framework.core.signal_adapter import enrich_signals, split_approved_vetoed

SYMBOL = "NQ"
print("Repo root:", ROOT)
print("CWD:", Path.cwd())
print("Config path:", CONFIG_PATH)

## 1. Load Data

In [ ]:
cfg = load_config(str(CONFIG_PATH))
loader = DataLoader(cfg)
df = loader.load_enriched(SYMBOL)

registry = FeatureRegistry(cfg)
df = registry.ensure_features(df, ["atr_14", "vix_regime", "chop_score"])

print(f"Loaded {len(df)} bars for {SYMBOL}")
display(df.tail())

## 2. Strategy Execution & Enrichment

In [ ]:
strategy = BoxReversionStrategy(ticker=SYMBOL)
raw_signals = strategy.hunt(df.last("120D"))

point_value = cfg.execution.point_value.get(SYMBOL, 20.0)
enriched = enrich_signals(
    raw_signals,
    df,
    strategy_name="box_reversion_deepdive",
    symbol=SYMBOL,
    point_value=point_value,
)

approved, vetoed = split_approved_vetoed(enriched)
print(f"Total Signals: {len(raw_signals)}")
print(f"Approved: {len(approved)} | Vetoed: {len(vetoed)}")

## 3. MFE/MAE Analysis

In [ ]:
if not approved.empty:
    results = compute_mfe_mae_rich(
        df,
        approved.head(50),
        max_forward_bars=120,
        horizons=[15, 30, 60, 120],
        atr_col="atr_14"
    )
    summary = summarize_mfe_mae_rich(results)
    display(Markdown("### Approved Signal Performance"))
    display(pd.DataFrame([summary]).T)
else:
    print("No approved signals found in the lookup window.")

## 4. Backtest & Quant Sheets

In [ ]:
engine = VectorizedBacktester()
metrics = engine.run(approved, df.last("120D"), {"ticker": SYMBOL})

if "equity_curve" in metrics and not metrics["equity_curve"].empty:
    run_id = f"DEEPDIVE_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{SYMBOL}"
    reporter = QuantReporter(run_id=run_id)
    
    curve = metrics["equity_curve"]
    returns = curve.pct_change().fillna(0)
    
    tear_sheet_path = reporter.generate_tear_sheet(returns, "DeepDive_v1")
    equity_plot_path = reporter.plot_equity_curve(curve, f"{SYMBOL}_Equity")
    
    display(Markdown(f"### Results Generated: {run_id}"))
    display(Markdown(f"- [Tear Sheet]({tear_sheet_path.as_uri()})"))
    display(Markdown(f"- [Equity Plot]({equity_plot_path.as_uri()})"))
else:
    print("Backtest yielded no trades; skipping reporting artifacts.")